In [1]:
import pandas as pd
import numpy as np
import pprint
from rdflib import Graph, URIRef, Literal
from rdflib.namespace import RDF, SKOS, DC

### Define vocabulary graph and declare concept scheme

In [2]:
voc = Graph()
concept_scheme = URIRef("https://vocabs.dariah.eu/category/")
voc.add((concept_scheme, RDF.type, SKOS.ConceptScheme))
voc.add((concept_scheme, SKOS.prefLabel, Literal("SSHOMP Resource Type", lang='en')))
voc.add((concept_scheme, DC.title, Literal("SSHOMP Resource Type", lang='en')))
voc.add((concept_scheme, DC.creator, Literal("Laure Barbot")))
voc.add((concept_scheme, DC.creator, Literal("Massimiliano Carloni")))
voc.add((concept_scheme, DC.creator, Literal("Matej Ďurčo")))
voc.add((concept_scheme, DC.creator, Literal("Vera Maria Charvát")))
voc.add((concept_scheme, DC.creator, Literal("Michael Kurzmeier")))

<Graph identifier=Ncdb11b227cc747f89d16ca1414d23406 (<class 'rdflib.graph.Graph'>)>

### Load vocabulary data from CSV

In [3]:
df = pd.read_csv('data.csv', skiprows=5, names=["top", "label_orig", "label_clean", "exact_match", "definition", "close_match", "comments"])

Replace all empty strings (or strings made just of spaces) with NaN, just to be sure

In [4]:
df = df.replace(r'^\s*$', np.nan, regex=True)

In [5]:
df.head()

,top,label_orig,label_clean,exact_match,definition,close_match,comments
0,Semantic Resource,NaN,semantic resource,NaN,"""machine-actionable formalisation (represented...",NaN,NaN
1,NaN,Metadata schema,metadata schema,NaN,"""Resources for defining and organising metadat...",NaN,NaN
2,NaN,Ontology,ontology,NaN,"""Resources for formalising knowledge within a ...",NaN,NaN
3,NaN,Vocabulary,vocabulary,NaN,"""Set of terms or concepts used within a specif...",NaN,NaN
4,NaN,Thesaurus,thesaurus,NaN,"""Structured, controlled vocabularies where ter...",NaN,NaN


In [6]:
seen_uris = []

top_concept = ""

for _, row in df.iterrows():
    # Create URIs from the cleaned labels, lowering the case and replacing spaces with hyphens
    uri = row["label_clean"].lower().replace(" ","-").strip()
    uri = URIRef("https://vocabs.dariah.eu/category/" + uri)
    voc.add((uri, RDF.type, SKOS.Concept))
    if uri in seen_uris:
        continue
    else:
        seen_uris.append(uri)
        # prefLabel
        voc.add((uri, SKOS.prefLabel, Literal(row["label_clean"], lang='en')))
        # exactMatch (does not handle lists)
        if pd.notna(row["exact_match"]):
            voc.add((uri, SKOS.exactMatch, URIRef(row["exact_match"])))
        # definition
        voc.add((uri, SKOS.definition, Literal(row["definition"])))
        # comments
        if pd.notna(row["comments"]):
            voc.add((uri, SKOS.editorialNote, Literal(row["comments"])))
        # associate with topConcept
        if pd.notna(row["top"]):
            voc.add((uri, SKOS.topConceptOf, concept_scheme))
            top_concept = uri
        else:
            voc.add((uri, SKOS.broader, top_concept))

for stmt in voc:
    pprint.pprint(stmt)

voc.serialize(destination="vocabulary.ttl")

(rdflib.term.URIRef('https://vocabs.dariah.eu/category/map'),
 rdflib.term.URIRef('http://www.w3.org/1999/02/22-rdf-syntax-ns#type'),
 rdflib.term.URIRef('http://www.w3.org/2004/02/skos/core#Concept'))
(rdflib.term.URIRef('https://vocabs.dariah.eu/category/knowledge-organization-system'),
 rdflib.term.URIRef('http://www.w3.org/2004/02/skos/core#definition'),
 rdflib.term.Literal('"All types of schemes for organizing information and promoting knowledge management. Knowledge organization systems include classification schemes that organize materials at a general level (such as books on a shelf), subject headings that provide more detailed access, and authority files that control variant versions of key information (such as geographic names and personal names). They also include less-traditional schemes, such as semantic artifacts, semantic networks and ontologies. https://www.clir.org/pubs/reports/pub91/1knowledge/" (http://purl.org/coar/resource_type/GSZA-Y7V7)'))
(rdflib.term.URIRef('h

<Graph identifier=Ncdb11b227cc747f89d16ca1414d23406 (<class 'rdflib.graph.Graph'>)>